# Part 1: ETL

- find sources
- merge data
- standardize (e.g. adjust for inflation, convert into uniform measures)
- transform into percentage changes
- save the data

# Sources

For our research, we will use the following websites as sources:
- FRED (https://fred.stlouisfed.org) for data regarding
    1. Copper https://fred.stlouisfed.org/series/PCOPPUSDM
    2. Iron ore https://fred.stlouisfed.org/series/PIORECRUSDM
    3. Lead https://fred.stlouisfed.org/series/PLEADUSDM
    4. Nickel https://fred.stlouisfed.org/series/PNICKUSDM
    5. Tin https://fred.stlouisfed.org/series/PTINUSDM
    6. Uranium https://fred.stlouisfed.org/series/PURANUSDM
    7. Zinc https://fred.stlouisfed.org/series/PZINCUSDM
- Macrotrends (https://www.macrotrends.net) for data regarding
    1. Crude oil https://www.macrotrends.net/1369/crude-oil-price-history-chart
    2. Natural gas https://www.macrotrends.net/2478/natural-gas-prices-historical-chart
    3. Silver https://www.macrotrends.net/1470/historical-silver-prices-100-year-chart
    4. Gold https://www.macrotrends.net/1333/historical-gold-prices-100-year-chart
    5. Platinum https://www.macrotrends.net/2540/platinum-prices-historical-chart-data
- Yahoo Finance (https://finance.yahoo.com) for data regarding
    1. Palladium https://finance.yahoo.com/quote/PA%3DF/history/?period1=906955200&period2=1778507301&filter=history&frequency=1mo

### Note: prices are given in USD

### Data extraction is done via directly downloading the data from the websites

In [1]:
import pandas as pd

df_copp=pd.read_csv('Data/FRED/Copper data (FRED).csv')
df_iror=pd.read_csv('Data/FRED/Iron ore data.csv')
df_lead=pd.read_csv('Data/FRED/Lead data.csv')
df_nickel=pd.read_csv('Data/FRED/Nickel data.csv')
df_ur=pd.read_csv('Data/FRED/Uranium data.csv')
df_zn=pd.read_csv('Data/FRED/Zinc data.csv')

df_coil=pd.read_csv('Data/MACROTRENDS/Crude oil data.csv')
df_gold=pd.read_csv('Data/MACROTRENDS/Gold data.csv')
df_natgas=pd.read_csv('Data/MACROTRENDS/Natural gas data.csv')
df_plat=pd.read_csv('Data/MACROTRENDS/Platinum data.csv')
df_silver=pd.read_csv('Data/MACROTRENDS/Silver data.csv')

import yfinance as yf

ticker='PA=F'

df_pall = yf.download(
    'PA=F',
    period='max',
    interval='1mo',
    auto_adjust=False,
    progress=False
)

C:\WinPython\WPy64-31050\python-3.10.5.amd64\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


### Data preparation (FRED)

In [2]:
#renaming
df_copp.columns=['Date','Copper']
df_iror.columns=['Date','Iron ore']
df_lead.columns=['Date','Lead']
df_nickel.columns=['Date','Nickel']
df_ur.columns=['Date','Uranium']
df_zn.columns=['Date','Zinc']

#date transformation
df_copp['Date'] = pd.to_datetime(df_copp['Date']).dt.to_period('M')
df_iror['Date'] = pd.to_datetime(df_iror['Date']).dt.to_period('M')
df_lead['Date'] = pd.to_datetime(df_lead['Date']).dt.to_period('M')
df_nickel['Date'] = pd.to_datetime(df_nickel['Date']).dt.to_period('M')
df_ur['Date'] = pd.to_datetime(df_ur['Date']).dt.to_period('M')
df_zn['Date'] = pd.to_datetime(df_zn['Date']).dt.to_period('M')

#merging fred dfs
df_fred=[df_copp,df_iror,df_lead,df_nickel,df_ur,df_zn]
merged_fred=df_fred[0]
for df in df_fred[1:]:
    merged_fred=merged_fred.merge(df, on='Date' ,how='inner')

### Data preparation (Yahoo finance)

In [3]:
#reindexing
df_pall=df_pall[['Close']].reset_index()

#renaming
df_pall.columns=['Date','Palladium']

#date transformation
df_pall['Date'] = pd.to_datetime(df_pall['Date']).dt.to_period('M')

#merging fred and yahoo
merged_fred_yahoo=merged_fred.merge(df_pall,on='Date',how='inner')

### Data preparation (Macrotrends)

In [4]:
#renaming
df_coil.columns=['Date','Crude oil']
df_gold.columns=['Date','Gold']
df_natgas.columns=['Date','Natural gas']
df_plat.columns=['Date','Platinum']
df_silver.columns=['Date','Silver']

#prepairing the data for platinum
df_plat['Date'] = pd.to_datetime(df_plat['Date'])
df_plat= (
    df_plat
    .assign(Month=df_plat['Date'].dt.to_period('M'))
    .groupby('Month')
    .mean(numeric_only=True)
    .reset_index()
    .rename(columns={'Month': 'Date'})
)

#date transformation
df_coil['Date'] = pd.to_datetime(df_coil['Date']).dt.to_period('M')
df_gold['Date'] = pd.to_datetime(df_gold['Date']).dt.to_period('M')
df_natgas['Date'] = pd.to_datetime(df_natgas['Date']).dt.to_period('M')
#df_plat['Date'] = pd.to_datetime(df_plat['Date']).dt.to_period('M')
df_silver['Date'] = pd.to_datetime(df_silver['Date']).dt.to_period('M')

#merging macrotrends
df_macrotrends = [df_coil,df_gold,df_natgas,df_plat,df_silver]
merged_macrotrends=df_macrotrends[0]
for df in df_macrotrends[1:]:
    merged_macrotrends=merged_macrotrends.merge(df,on='Date',how='inner')

### Merging the data together

In [5]:
#merging
data=merged_fred_yahoo.merge(merged_macrotrends,on='Date',how='inner')

#permuting
data=data[['Date','Gold','Silver','Platinum','Palladium'
           ,'Copper','Iron ore','Lead','Nickel','Zinc'
           ,'Uranium','Crude oil','Natural gas']]

### Because the dataset combines commodities from different sources, inner joins were used to keep only months where all commodity prices were available.

In [6]:
data.head(5)

,Date,Gold,Silver,Platinum,Palladium,Copper,Iron ore,Lead,Nickel,Zinc,Uranium,Crude oil,Natural gas
0,1998-09,294.10,5.39,359.690476,281.149994,1646.772692,13.41,519.454529,4100.909180,999.636353,10.233333,16.19,2.22
1,1998-10,293.10,5.06,342.306818,277.000000,1585.499925,13.41,493.204559,3870.454590,941.363647,9.605556,14.48,2.00
2,1998-12,287.45,5.01,349.954545,332.149994,1475.763094,13.41,500.605255,3865.763184,961.578918,8.487500,12.14,1.95
3,1999-01,285.65,5.24,353.934211,336.500000,1432.000028,11.93,491.625000,4264.000000,931.750000,9.012500,12.81,1.83
4,1999-02,286.80,5.58,364.236842,350.600006,1412.950000,11.93,512.950000,4623.250000,1017.600000,9.200000,12.31,1.63


### Now we are ought to uniform the data since the prices are given for different measure units (e.g. gramm, troy ounce, etc...) and adjust for inflation if necessary. We create a table to get an overview:

|Property\Commodity|Gold|Silver|Platinum|Palladium|Copper|Iron ore|Lead|Nickel|Zinc|Uranium|Crude oil|Natural gas|
|--|---|------|--------|---------|------|--------|----|------|----|-------|---------|-----------|
|Inflation adjusted|Yes|Yes|No|No|No|No|No|No|No|No|Yes|No|
|Initial measure unit|Troy ounce (31,1034768 g)|Troy ounce (31,1034768 g)|Troy ounce (31,1034768 g)|Troy ounce (31,1034768 g)|Metric ton (1,000,000 g)|Metric ton (1,000,000 g)|Metric ton (1,000,000 g)|Metric ton (1,000,000 g)|Metric ton (1,000,000 g)|Pound (453.59237 g)|Barrel (136,000 g)|MMBtu (19,000 g)|


### For energy commodities, measure unit conversion is approximate and mainly used for normalization rather than exact valuation

In [7]:
#translate the table into a data frame
dict0={'Gold':[1,31.1034768],'Silver':[1,31.1034768],'Platinum':[0,31.1034768],
      'Palladium':[0,31.1034768],'Copper':[0, 1000000],'Iron ore':[0,1000000],
      'Lead':[0,1000000],'Nickel':[0,1000000],'Zinc':[0,1000000],'Uranium':[0,453.59237],
      'Crude oil':[1,136000],'Natural gas':[0,19000]}

table0=pd.DataFrame(dict0)

In [8]:
#adjusting for the measure units: translating everything into gramm
for name in table0.columns:
    data[name]=data[name]/table0[name].iloc[1]

### Adjustment for inflation is done via $price(t) \cdot \frac{BaseCPI}{CPI(t)}$ with the latest available base date, that is March 2026.

In [9]:
#adjusting for inflation
CPI=pd.read_csv('Data/FRED/CPI.csv')
CPI.columns=['Date','CPI']
CPI['Date']=pd.to_datetime(CPI['Date']).dt.to_period("M")
CPI['CPI']=pd.to_numeric(CPI['CPI'])
CPI=CPI.dropna()

Base_CPI=CPI.iloc[-1,1]

CPI['inflation_coefficient']=Base_CPI/CPI['CPI']

data=data.merge(CPI[['Date', 'inflation_coefficient']],on='Date',how='inner')

for name in table0.columns:
    if table0[name].iloc[0] == 0:
        data[name] = data[name]*data['inflation_coefficient']

data=data.drop(columns='inflation_coefficient')

In [10]:
data=data.dropna()

### Constructing monthly returns data

In [11]:
#creating return data
data_return=data.drop(columns=['Date']).pct_change().dropna().reset_index(drop=True)
data_return.insert(0,'Date',data['Date'].iloc[1:].reset_index(drop=True))

# Data check

### For prices

In [12]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 277 entries, 0 to 276
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype    
---  ------       --------------  -----    
 0   Date         277 non-null    period[M]
 1   Gold         277 non-null    float64  
 2   Silver       277 non-null    float64  
 3   Platinum     277 non-null    float64  
 4   Palladium    277 non-null    float64  
 5   Copper       277 non-null    float64  
 6   Iron ore     277 non-null    float64  
 7   Lead         277 non-null    float64  
 8   Nickel       277 non-null    float64  
 9   Zinc         277 non-null    float64  
 10  Uranium      277 non-null    float64  
 11  Crude oil    277 non-null    float64  
 12  Natural gas  277 non-null    float64  
dtypes: float64(12), period[M](1)
memory usage: 28.3 KB


In [13]:
data.isna().sum()

Date           0
Gold           0
Silver         0
Platinum       0
Palladium      0
Copper         0
Iron ore       0
Lead           0
Nickel         0
Zinc           0
Uranium        0
Crude oil      0
Natural gas    0
dtype: int64

In [14]:
data.describe()

,Gold,Silver,Platinum,Palladium,Copper,Iron ore,Lead,Nickel,Zinc,Uranium,Crude oil,Natural gas
count,277.000000,277.000000,277.000000,277.000000,277.000000,277.000000,277.000000,277.000000,277.000000,277.000000,277.000000,277.000000
mean,38.388564,0.560894,47.855311,35.969382,0.008073,0.000103,0.002296,0.022189,0.002930,0.110248,0.000449,0.000332
std,25.026073,0.378696,17.475245,22.720312,0.003185,0.000067,0.000922,0.010794,0.001004,0.066098,0.000190,0.000199
min,8.254704,0.133104,22.178237,9.042522,0.002562,0.000023,0.000762,0.007767,0.001364,0.029442,0.000089,0.000086
25%,14.035408,0.231164,34.492769,19.285112,0.005574,0.000047,0.001645,0.015374,0.002213,0.066703,0.000294,0.000198
50%,39.686882,0.528880,42.240590,31.728240,0.008737,0.000097,0.002386,0.019723,0.002862,0.099797,0.000449,0.000270
75%,53.394352,0.755350,62.150412,41.903248,0.010402,0.000140,0.002894,0.025383,0.003461,0.141856,0.000588,0.000421
max,156.424635,3.663668,101.265680,117.804423,0.014708,0.000279,0.005201,0.082724,0.007160,0.478653,0.000936,0.001298


In [15]:
print(data['Date'].min(), data['Date'].max())

1998-09 2026-01


In [16]:
print(data.shape)

(277, 13)


### For returns

In [17]:
data_return.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 276 entries, 0 to 275
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype    
---  ------       --------------  -----    
 0   Date         276 non-null    period[M]
 1   Gold         276 non-null    float64  
 2   Silver       276 non-null    float64  
 3   Platinum     276 non-null    float64  
 4   Palladium    276 non-null    float64  
 5   Copper       276 non-null    float64  
 6   Iron ore     276 non-null    float64  
 7   Lead         276 non-null    float64  
 8   Nickel       276 non-null    float64  
 9   Zinc         276 non-null    float64  
 10  Uranium      276 non-null    float64  
 11  Crude oil    276 non-null    float64  
 12  Natural gas  276 non-null    float64  
dtypes: float64(12), period[M](1)
memory usage: 28.2 KB


In [18]:
data_return.isna().sum()

Date           0
Gold           0
Silver         0
Platinum       0
Palladium      0
Copper         0
Iron ore       0
Lead           0
Nickel         0
Zinc           0
Uranium        0
Crude oil      0
Natural gas    0
dtype: int64

In [19]:
data_return.describe()

,Gold,Silver,Platinum,Palladium,Copper,Iron ore,Lead,Nickel,Zinc,Uranium,Crude oil,Natural gas
count,276.000000,276.000000,276.000000,276.000000,276.000000,276.000000,276.000000,276.000000,276.000000,276.000000,276.000000,276.000000
mean,0.011501,0.016162,0.006711,0.010302,0.007342,0.010283,0.004941,0.007402,0.004327,0.007505,0.012073,0.023749
std,0.051091,0.103018,0.069458,0.113490,0.069253,0.105708,0.071523,0.096478,0.072489,0.079445,0.119409,0.224314
min,-0.187848,-0.283951,-0.247597,-0.288855,-0.292148,-0.473337,-0.270491,-0.335960,-0.246610,-0.227725,-0.571046,-0.537296
25%,-0.022570,-0.046749,-0.031376,-0.058372,-0.025150,-0.011942,-0.028939,-0.048324,-0.038451,-0.026019,-0.058844,-0.099027
50%,0.006431,0.004906,0.004453,0.020598,0.003697,-0.001767,0.006070,0.002601,0.002566,0.000183,0.020565,0.001455
75%,0.039052,0.069733,0.039261,0.076043,0.038087,0.041078,0.040157,0.064214,0.042051,0.038168,0.072608,0.093804
max,0.183057,0.590431,0.391349,0.557622,0.355906,0.715965,0.275812,0.465559,0.295191,0.383726,0.849714,1.282227


In [20]:
print(data_return['Date'].min(), data_return['Date'].max())

1998-10 2026-01


In [21]:
print(data_return.shape)

(276, 13)


In [22]:
data_return.to_csv('Non-food commodities - Part 1 (data in pct_change).csv',index=False)

data_return.to_excel('Non-food commodities - Part 1 (data in pct_change).xlsx',index=False)

data.to_csv('Non-food commodities - Part 1 (data in price levels).csv',index=False)

data.to_excel('Non-food commodities - Part 1 (data in price levels).xlsx',index=False)

C:\Users\ibrag\AppData\Local\Temp\ipykernel_7896\1215055740.py:3: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  data_return.to_excel('Non-food commodities - Part 1 (data in pct_change).xlsx',index=False)
C:\Users\ibrag\AppData\Local\Temp\ipykernel_7896\1215055740.py:7: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  data.to_excel('Non-food commodities - Part 1 (data in price levels).xlsx',index=False)


# Data description:

### Data name: 'Non-food commodities - Part 1 (data in pct_change)'
- Interval: monthly data
- Entries: monthly returns
- Time period: from 1998-09 to 2026-01

### Data name: 'Non-food commodities - Part 1 (data in price levels)'
- Interval: monthly data
- Entries: monthly prices per gram
- Time period: from 1998-10 to 2026-01

# Now we are finally prepared for data analysis